In [16]:
import pandas as pd
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize
import string

In [17]:
def preprocess(text):
    if pd.isna(text):
        return []
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in string.punctuation]
    return tokens

What data is relevant for the task of finding similar episodes based on descriptions?
What data is relevant for finding an episode suitable for user query?

The episode descriptions from the dw_imdb dataset are relevant for both tasks, as they provide textual information about each episode that can be used to compute similarity. They are also more useful since we'll be only working with the series reboot starting 2005 and not the "old" Doctor Who.

The scripts from the dw_scripts dataset may also be relevant if we want to analyze the content of the episodes in more detail, but for a basic similarity comparison, the descriptions should suffice.

In [18]:
import json

# Открыть JSON-файл и загрузить в словарь
with open("dw_data/document_corpus_dw.json", "r", encoding="utf-8") as f:
    document_corpus = json.load(f)

with open("dw_data/inverted_index.json", "r", encoding="utf-8") as f:
    inverted_index = json.load(f)

In [19]:
def boolean_search(query: str, index: dict):
    query_tokens = preprocess(query)
    if not query_tokens:
      return []
    results = set(index.get(query_tokens[0], []))

    for token in query_tokens[1:]:
      results = results.union(index.get(token, []))

    return list(results)[:5]

In [20]:
from rank_bm25 import BM25Okapi

# Подготовка корпуса для BM25
# Сохраняем ключи в том же порядке, в котором будут токенизированные описания
doc_keys = list(document_corpus.keys())
tokenized_corpus = [preprocess(document_corpus[k]['description']) for k in doc_keys]

# Создаём BM25 объект
bm25 = BM25Okapi(tokenized_corpus)

# Функция поиска
def bm25_search(query, n=5):
    query_tokens = preprocess(query)
    scores = bm25.get_scores(query_tokens)
    
    # Индексы топ-n документов
    ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    
    results = []
    for i in ranked_indices[:n]:
        doc_id = doc_keys[i]  # берём ключ из согласованного списка
        results.append({
            'episodeid': doc_id,
            'title': document_corpus[doc_id]['title'],
            'score': scores[i],
            'snippet': document_corpus[doc_id]['description'][:300]  # первые 300 символов
        })
    return results

In [21]:
from rank_bm25 import BM25Okapi

# Prepare corpus for BM25
tokenized_corpus = [preprocess(doc['description']) for doc in document_corpus.values()]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query, n=5):
    query_tokens = preprocess(query)
    scores = bm25.get_scores(query_tokens)
    ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    
    results = []
    for i in ranked_indices[:n]:
        doc_id = list(document_corpus.keys())[i]
        results.append({
            'episodeid': doc_id,
            'title': document_corpus[doc_id]['title'],
            'score': scores[i],
            'snippet': document_corpus[doc_id]['description'][:300]  # first 300 chars of episode
        })
    return results

In [22]:
my_queries = ["Doctor fights with Weeping Angels and wants to save Amy",
              "Doctor and Clara are in nineteenth century",
              "Doctor meet River Song for the first time",
              "Doctor and Donna encounter Daleks and Davros",
              "Doctor and Rose end up in a parallel universe",
              "Doctor meets van Gogh",
              "Doctor and Martha face time paradox creatures or similar threats",
              "Doctor and Bill encounter Cybermen",
              "Paternoster Gang and Doctor and Vastra and Strax",
              "Doctor and Rose meet her father Pete Tyler"
              ]

In [23]:
BOLD = "\033[1m"
RESET = "\033[0m"

for query in my_queries:
    print(f"\n{BOLD}Query '{query}'{RESET}")
    hits = boolean_search(query, inverted_index)
    
    # Keep only hits present in document_corpus
    hits = [id for id in hits if id in document_corpus]
    
    print(f"\nBoolean hits:")
    for i, id in enumerate(hits):
        episode = document_corpus.get(id)
        print(f"{i+1}: {id} - {episode['title']}\n{episode['description']}")
    
    print("\nBM25 Results:")
    results = bm25_search(query, n=5)
    for i, res in enumerate(results):
        print(f"Rank {i+1}: {res['episodeid']} - {res['title']} (score={res['score']:.2f})")
        print(res['snippet'])
        print("*-----" * 50)


Query 'Doctor fights with Weeping Angels and wants to save Amy'

Boolean hits:
1: 5x11 - The Lodger
No sooner does the TARDIS land on Earth that it leaves again - but without the Doctor who had just stepped outside. The Doctor soon finds himself at the home of Craig Owens, who has been advertising for a lodger. There's clearly something odd in the house with people being lured to the upstairs room, but never reappearing. The Doctor is having a good time of it and is having a bit of fun; he proves to be a rather good football player. Craig the landlord is very much in love with Sophie but can't quite bring himself to tell her. The Doctor tries to help them out. But ...
2: 1x8 - Father's Day
All of her life, Rose's mother told how great a man her father Pete was. He was killed by a hit and run driver when Rose was just a baby and among her many regrets was that her father died alone, just lying on the street. Rose asks the Doctor to take her back to that day. Rather than just be with hi

In [24]:
for query in my_queries:
    results = bm25_search(query, n=5)
    for res in results:
        print(f"{res['episodeid']},", end=' ')
    print("\n")

7x5, 5x4, 5x5, 6x11, 6x0, 

5x6, 2x2, 7x12, 2x4, 4x3, 

7x5, 6x1, 6x8, 9x13, 5x5, 

2x13, 1x11, 4x13, 7x1, 4x3, 

2x5, 4x2, 8x4, 10x1, 6x4, 

5x10, 1x1, 6x0, 1x5, 8x3, 

9x0, 5x10, 9x6, 4x11, 5x3, 

1x11, 2x6, 10x8, 10x9, 2x13, 

7x12, 1x5, 2x6, 2x1, 2x2, 

1x8, 1x1, 2x2, 2x1, 7x12, 

